In [ ]:
from collections import defaultdict
import cv2
import numpy as np
import supervision as sv

from ultralytics import YOLO

model = YOLO('yolov8n.pt')

video_path = "traffic3.mp4"
cap = cv2.VideoCapture(video_path)

track_history = defaultdict(lambda: [])

# Create a dictionary to keep track of objects that have crossed the line for each region
crossed_objects = {1: defaultdict(lambda: False), 2: defaultdict(lambda: False), 3: defaultdict(lambda: False), 4: defaultdict(lambda: False)}

# Coordinates for ROIs, each variable is a tuple for (x,y)
# Naming convention will be top of each region i.e. tl1 ==> top left region 1,  tr1 ==> top right region 1, tl2 ==> top left region 2,  tr2 ==> top right region 2. 
# setting the corners of the regions

tl1 = (325, 20)
tr1 = (1240, 20)

tl2 = (220, 120)
tr2 = (1240, 120)

tl3 = (127, 280)
tr3 = (1270, 280)

tl4 = (20, 497)
tr4 = (1270, 497)

bl4 = (0, 700)
br4 = (1270, 700)

# Create bounding boxes for each region
region_boxes = {
    1: [tl1, tr1, bl4, br4],
    2: [tl2, tr2, tr3, tl3],
    3: [tl3, tr3, tr4, tl4],
    4: [tl4, tr4, br4, bl4]
}

# Function to check if a point is inside a polygon
def is_point_in_polygon(point, polygon):
    x, y = point
    n = len(polygon)
    inside = False
    p1x, p1y = polygon[0]
    for i in range(1, n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xints = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xints:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()
    if success:
        # Run YOLOv8 tracking on the frame, persisting tracks between frames
        results = model.track(frame, persist=True)
        for result in results:
            if result.boxes is None or result.boxes.id is None:
                continue
            else:
                # Get the boxes and track IDs
                boxes = result.boxes.xywh.cpu()
                track_ids = result.boxes.id.cpu().numpy().astype(int)

                # Visualize the results on the frame
                annotated_frame = result.plot()

                # Plot the tracks
                for box, track_id in zip(boxes, track_ids):
                    x, y, w, h = box
                    track = track_history[track_id]
                    track.append((float(x), float(y)))  # x, y center point
                    if len(track) > 30:  # retain 30 tracks for 30 frames
                        track.pop(0)

                    # Check if the object is inside any of the regions
                    for region_id, vertices in region_boxes.items():
                        if is_point_in_polygon((x, y), vertices):
                            crossed_objects[region_id][track_id] = True

            # Draw polygons around the vertices
            cv2.polylines(annotated_frame, [np.array([tl1, tr1, tr2, tl2], dtype=np.int32)], True, (255, 0, 0), 2)
            cv2.polylines(annotated_frame, [np.array([tl2, tr2, tr3, tl3], dtype=np.int32)], True, (255, 0, 0), 2)
            cv2.polylines(annotated_frame, [np.array([tl3, tr3, tr4, tl4], dtype=np.int32)], True, (255, 0, 0), 2)
            cv2.polylines(annotated_frame, [np.array([tl4, tr4, br4, bl4], dtype=np.int32)], True, (255, 0, 0), 2)

            # Write the count of objects on each frame
            count_text = f"Objects crossed region 1: {len(crossed_objects[1])}"
            cv2.putText(annotated_frame, count_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            count_text = f"Objects crossed region 2: {len(crossed_objects[2])}"
            cv2.putText(annotated_frame, count_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            count_text = f"Objects crossed region 3: {len(crossed_objects[3])}"
            cv2.putText(annotated_frame, count_text, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            count_text = f"Objects crossed region 4: {len(crossed_objects[4])}"
            cv2.putText(annotated_frame, count_text, (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Display the annotated frame
            cv2.imshow("YOLOv8 Tracking", annotated_frame)

            # Break the loop if 'q' is pressed
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    else:
        # Break the loop if the end of the video is reached
        break
        

        
car_count = [len(crossed_objects[1]), len(crossed_objects[2]), len(crossed_objects[3]), len(crossed_objects[4])]
print("car count", car_count)
    


vertices = [
    [tl1, tr1, tr2, tl2],
    [tl2, tr2, tr3, tl3],
    [tl3, tr3, tr4, tl4],
    [tl4, tr4, br4, bl4]
]

merged_vertices = []

def merge_regions(car_count, vertices):
    merged_arr = []
    merged_vertices = []
    i = 0
    while i < len(car_count):
        # Check if we can merge with next element
        if i < len(car_count) - 1 and 0.7 * car_count[i] <= car_count[i + 1] <= 1.3 * car_count[i]:
            # Merge the two adjacent elements by averaging
            merged_arr.append((car_count[i] + car_count[i + 1]) / 2)
            merged_vertices.append([vertices[i][0], vertices[i][1], vertices[i + 1][2], vertices[i + 1][3]])
            # Skip the next element as it's already merged
            i += 2
        else:
            # If cannot merge, just append the current element
            merged_arr.append(car_count[i])
            merged_vertices.append(vertices[i])
            i += 1
    return merged_arr, merged_vertices

car_count, merged_vertices = merge_regions(car_count, vertices)
# merge_regions(car_count)

print("====ORiginal====")
for e in vertices:
    print(e,end="\n") 
print("=====Merged=======")
# print(merged_vertices)
for e in merged_vertices:
    print(e,end="\n") 
    
    
cap.release()

# Wait for a while before displaying the merged regions
cv2.waitKey(1000)

# Now let's display the video again with merged regions and their counts

# Video capture
cap = cv2.VideoCapture('traffic.mp4')

while True:
    ret, frame = cap.read()

    if not ret:
        # If not, either end of the video or an error occurred
        print("End of video or error occurred")
        break
    
    for i in range(len(merged_vertices)):
        # Convert vertices to numpy array
        vertices_array = np.array(merged_vertices[i], np.int32)
        cv2.polylines(frame, [vertices_array], isClosed=True, color=(0, 0, 255), thickness=2)




    # Display car count for each merged region
    for i, count in enumerate(car_count):
        cv2.putText(frame, f'Merged Car Count R{i+1}: {count}', (10, 210 + 30*i), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Display frame with merged regions
    cv2.imshow('Video with Merged Regions', frame)

    k = cv2.waitKey(30) & 0xff
    if k == 27:
        break


cap.release()
cv2.destroyAllWindows()



0: 384x640 1 person, 7 cars, 350.7ms
Speed: 5.7ms preprocess, 350.7ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.



0: 384x640 1 person, 7 cars, 138.1ms
Speed: 1.6ms preprocess, 138.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7 cars, 128.8ms
Speed: 1.9ms preprocess, 128.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7 cars, 152.0ms
Speed: 2.0ms preprocess, 152.0ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7 cars, 154.1ms
Speed: 3.2ms preprocess, 154.1ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)



2024-11-19 19:32:25.519 python[36840:5166721] +[IMKClient subclass]: chose IMKClient_Legacy
2024-11-19 19:32:25.519 python[36840:5166721] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


0: 384x640 1 person, 8 cars, 141.2ms
Speed: 2.4ms preprocess, 141.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 144.8ms
Speed: 2.7ms preprocess, 144.8ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 144.0ms
Speed: 2.7ms preprocess, 144.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 248.7ms
Speed: 3.9ms preprocess, 248.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 244.4ms
Speed: 3.8ms preprocess, 244.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 187.6ms
Speed: 2.8ms preprocess, 187.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 185.2ms
Speed: 4.7ms preprocess, 185.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 218.5ms
Speed: 2.8ms preprocess, 218.5ms inferenc